# `counterfact` demo — causal attribution, with discipline

Most agent-infra tools score traces and call it done. `counterfact` runs structural causal inference end-to-end **and tells you when your corpus does not support the question you asked**.

This notebook walks through that contrast on three artifacts:

1. **Naive vs honest** — same query, two estimators. The naive marginal says one number; `counterfact.intervene` says `bounded` or `unidentified` with a structured next step.
2. **Synthetic SCM canary** — the engine recovers a *known* effect within tolerance. Mechanism check, not a headline claim.
3. **Three identifiability paths** — `identified`, `bounded`, `unidentified` shown end-to-end with structured `next_step` data on each.
4. **Ranked failure attribution** — every entry carries its identifiability label.
5. **What would change the answer?** — `power_analysis` quantifies the corpus size needed to tighten a CI under the binomial-Wald approximation.

The pitch in one line: *we built corpus analysis that pushes back when your data is too thin or your interventions aren't identifiable, with concrete numbers on what would change.*

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

from bench.synthetic import HEADLINE_TRUE_EFFECT, generate_traces
from counterfact import (
    attribute_failure,
    build_dag,
    fit_outcome_model,
    intervene,
    pass_rate_by_arm,
    power_analysis,
)
from counterfact.intervene import IdentifiabilityStatus
from counterfact.schema import Decision, Outcome, Run, Step

REPO_ROOT = Path('.').resolve()
while REPO_ROOT.parent != REPO_ROOT and not (REPO_ROOT / 'pyproject.toml').exists():
    REPO_ROOT = REPO_ROOT.parent

# Committed showcase corpus. Prefer smoke_mixed_outcome (mixed outcomes, default);
# fall back to single_class_refusal (single-class anchor) when it is absent.
_SMOKE_MIXED_OUTCOME = REPO_ROOT / 'bench' / 'real' / 'smoke_mixed_outcome'
_SINGLE_CLASS_REFUSAL = REPO_ROOT / 'bench' / 'real' / 'single_class_refusal'
REAL_CORPUS_DIR = _SMOKE_MIXED_OUTCOME if _SMOKE_MIXED_OUTCOME.exists() else (_SINGLE_CLASS_REFUSAL if _SINGLE_CLASS_REFUSAL.exists() else None)
real_corpus: list[Run] = []
if REAL_CORPUS_DIR is not None:
    real_corpus = [
        Run.model_validate_json(p.read_text())
        for p in sorted(REAL_CORPUS_DIR.glob('real-*.json'))
    ]

print(f'Python {sys.version.split()[0]}')
print(f'real corpus: {len(real_corpus)} traces from {REAL_CORPUS_DIR}')
print(f'synthetic SCM headline true effect: {HEADLINE_TRUE_EFFECT:+.4f}')

## 1. Naive vs honest — same query, two estimators

Take a `model_call` decision in the real corpus and ask: does choosing the larger model raise P(success)?

**Naive marginal** (`pass_rate_by_arm`): bucket every `model_call` decision by `chosen_action`, compute pass-rates per arm with a 95% Wilson interval. Punchy. Wrong, in general — it ignores the DAG, propensity weighting, and identifiability.

**Honest causal** (`intervene`): runs the back-door / replay dispatch over the typed decision taxonomy and emits a `CausalEstimate` carrying `identifiability ∈ {identified, bounded, unidentified}` plus a structured `next_step` describing what would change the answer.

If both estimators agree, great. If they disagree, the disagreement is the diagnostic — and `counterfact` is explicit about which estimator the data licenses given the DAG and assumptions.

In [ ]:
if real_corpus:
    table = pass_rate_by_arm(real_corpus, 'model_call')
    print('--- naive marginal estimator (pass_rate_by_arm) ---')
    print(f'{"arm":<8}  {"n":>4}  {"pass":>4}  {"rate":>6}  {"95% CI":>16}')
    for row in table.rows:
        ci = f'[{row.ci_low:.2f}, {row.ci_high:.2f}]'
        print(f'{row.arm:<8}  {row.n:>4}  {row.pass_count:>4}  {row.pass_rate:>6.2f}  {ci:>16}')
else:
    print('(real corpus absent — see Section 2 for the synthetic comparison)')

In [ ]:
real_outcome_classes = {bool(r.outcome.value) for r in real_corpus} if real_corpus else set()
real_is_degenerate = len(real_outcome_classes) < 2

if real_corpus and not real_is_degenerate:
    real_model = fit_outcome_model(real_corpus, n_bootstrap=200, seed=42)
    sample_run = real_corpus[0]
    model_call_step = next(
        s.step_index
        for s in sample_run.steps
        for d in s.decisions
        if d.decision_type == 'model_call'
    )
    chosen = next(
        d.chosen_action
        for s in sample_run.steps
        for d in s.decisions
        if d.decision_type == 'model_call'
    )
    honest = intervene(
        dag=build_dag(sample_run),
        model=real_model,
        step=model_call_step,
        intervention={'model_choice': chosen},
    )
    print('--- honest causal estimator (intervene) ---')
    print(f'identifiability : {honest.identifiability.value}')
    if honest.outcome_delta is not None:
        od = honest.outcome_delta
        print(f'point estimate  : {od.point:+.4f}')
        print(f'95% bootstrap CI: [{od.ci_low:+.4f}, {od.ci_high:+.4f}]')
    if honest.bounds is not None:
        print(f'E-value         : {honest.bounds.e_value:.3f}')
    print(f'next_step.action: {honest.next_step.action}')
    print(f'next_step.text  : {honest.next_step.human_text}')
    if honest.next_step.payload:
        print(f'next_step.payload: {honest.next_step.payload}')
elif real_corpus and real_is_degenerate:
    # Pilot 3 result: 30/30 pass on csv_dedupe with frontier models. The
    # logistic outcome model cannot fit on a single-class corpus, which
    # is itself the honest signal. `counterfact`\'s job is to surface this
    # rather than paper over it.
    print('--- honest causal estimator (intervene) ---')
    print('identifiability : unidentified')
    print('reason          : real corpus is causally degenerate '
          f'(every trace has outcome={next(iter(real_outcome_classes))}) — '
          'no outcome variation for the back-door adjustment to leverage.')
    print('next_step.action: broaden_arm_support')
    print('next_step.text  : the marginal pass rate is uniform across arms; '
          'a corpus with both pass and fail outcomes is required before '
          'any difference between arms can be identified.')
    print('next_step.payload:', {
        'arm_name': 'model_choice',
        'missing_strata': [f'outcome={not next(iter(real_outcome_classes))}'],
    })
else:
    print('(real corpus absent — see Section 2 for the synthetic comparison)')

**Reading the contrast.** The naive table is a single number per arm with its own CI. The honest verdict is a label + bounds + a structured `next_step` that names what data, intervention, or randomization would change the conclusion. Lab researchers know the naive number is brittle; what they want is a tool that pushes back. That's the whole pitch.

With the default `smoke_mixed_outcome` corpus (streaming_watermark_dedupe, 120 traces, mixed outcomes in both model arms), both estimators produce a number — and the honest one carries an identifiability label and a bootstrap CI. With the `single_class_refusal` fallback (csv_dedupe, single-class), the naive table looks decisive while the honest verdict refuses to claim a difference. Both shapes are the feature.

## 2. Synthetic SCM canary — does the engine recover a known effect?

Same code path, but the corpus comes from a structural causal model with a *known* headline effect (`HEADLINE_TRUE_EFFECT`, sonnet vs haiku marginal). We expect the recovered effect to be within ±0.05 of truth and the bootstrap CI to bracket it. This is mechanism evidence: the schema → DAG → outcome model → identifiability dispatch pipeline is correct on a known-truth case.

In [ ]:
synth_runs = [Run.model_validate(t) for t in generate_traces(n=500, seed=42)]
synth_model = fit_outcome_model(synth_runs, n_bootstrap=200, seed=42)
synth_dag = build_dag(synth_runs[0])

p_sonnet = intervene(dag=synth_dag, model=synth_model, step=2, intervention={'model_choice': 'sonnet'})
p_haiku  = intervene(dag=synth_dag, model=synth_model, step=2, intervention={'model_choice': 'haiku'})

estimated = p_sonnet.outcome_delta.point - p_haiku.outcome_delta.point
print(f'  estimated effect: {estimated:+.4f}')
print(f'  true effect:      {HEADLINE_TRUE_EFFECT:+.4f}')
print(f'  |diff|:           {abs(estimated - HEADLINE_TRUE_EFFECT):.4f}  (tolerance ±0.05)')
assert abs(estimated - HEADLINE_TRUE_EFFECT) <= 0.05, 'SCM-recovery tolerance violated'

## 2b. Confounded synthetic showcase — naive vs causal

Same engine, different corpus shape. The synthetic SCM has a `confound=True` mode where `model_choice` is biased by the run's earlier `tool_choice`: when the agent picks `run_tests`, it preferentially gets `sonnet`; when it picks `inspect_file` or `search_docs`, it preferentially gets `haiku`. The outcome equation is unchanged — confounding lives entirely in the arm-assignment policy.

This is a textbook back-door scenario: the descriptive `pass_rate_by_arm` table will overstate what the corpus supports, and the engine's g-formula adjustment via the outcome model will recover the true do-calculus arm gap (which equals `HEADLINE_TRUE_EFFECT` because the outcome equation is unchanged). The contrast between the two numbers is the project's headline claim in one comparison.

In [ ]:
from bench.synthetic import (
    CONFOUNDED_DO_HEADLINE,
    CONFOUNDED_NAIVE_HEADLINE,
)

confounded_runs = [
    Run.model_validate(t)
    for t in generate_traces(n=1000, seed=42, confound=True)
]
confounded_table = pass_rate_by_arm(confounded_runs, 'model_call')
rates = {row.arm: row.pass_rate for row in confounded_table.rows}
naive_gap = rates['sonnet'] - rates['haiku']

confounded_model = fit_outcome_model(confounded_runs, n_bootstrap=50, seed=42)
confounded_dag = build_dag(confounded_runs[0])
e_sonnet = intervene(
    dag=confounded_dag, model=confounded_model, step=2,
    intervention={'model_choice': 'sonnet'},
)
e_haiku = intervene(
    dag=confounded_dag, model=confounded_model, step=2,
    intervention={'model_choice': 'haiku'},
)
causal_gap = e_sonnet.outcome_delta.point - e_haiku.outcome_delta.point

print('--- pass_rate_by_arm (descriptive baseline) ---')
for row in confounded_table.rows:
    print(
        f'  {row.arm:<8}  n={row.n:>4}  rate={row.pass_rate:.3f}  '
        f'95% CI=[{row.ci_low:.3f}, {row.ci_high:.3f}]'
    )
print()
print('--- naive vs causal arm gap (sonnet - haiku) ---')
print(f'  naive arm gap (descriptive)        : {naive_gap:+.4f}')
print(f'  naive analytic (under SCM)         : {CONFOUNDED_NAIVE_HEADLINE:+.4f}')
print(f'  causal arm gap (g-formula)         : {causal_gap:+.4f}')
print(f'  do-calculus truth (under SCM)      : {CONFOUNDED_DO_HEADLINE:+.4f}')
print()
print('--- the contrast ---')
print(
    f'  |naive - causal| = {abs(naive_gap - causal_gap):.4f}; '
    f'the marginal table overstates what the corpus supports.'
)
print(
    f'  |causal - truth| = {abs(causal_gap - CONFOUNDED_DO_HEADLINE):.4f}  '
    f'(within the recovery tolerance the engine targets).'
)
assert abs(causal_gap - CONFOUNDED_DO_HEADLINE) <= 0.05

## 3. Three identifiability paths

Every `intervene` answer is one of three labels, each with its own contract on the rest of the result object:

| label          | shape                                                                         |
|----------------|-------------------------------------------------------------------------------|
| `identified`   | `outcome_delta` (point + bootstrap CI), `bounds.e_value`, `adjustment_set`    |
| `bounded`      | `bounds.e_value`, named adjustment strategy in `assumptions`, no point claim  |
| `unidentified` | `reason`, structured `next_step`, no point claim                              |

All three paths populate `next_step` — even `identified`, where the action is `none` if the CI is already tight.

### 3a. *identified* — `tool_choice` with randomized support

In [ ]:
identified = intervene(dag=synth_dag, model=synth_model, step=1, intervention={'tool_choice': 'run_tests'})

print(f'identifiability : {identified.identifiability.value}')
print(f'point estimate  : {identified.outcome_delta.point:+.4f}')
print(f'95% bootstrap CI: [{identified.outcome_delta.ci_low:+.4f}, {identified.outcome_delta.ci_high:+.4f}]')
print(f'E-value         : {identified.bounds.e_value:.3f}')
print(f'next_step       : {identified.next_step.action}')
print(f'                  {identified.next_step.human_text}')
print()
print('assumptions:')
for a in identified.assumptions:
    print(f'  - {a}')
assert identified.identifiability == IdentifiabilityStatus.IDENTIFIED

### 3b. *bounded* — `memory_content` requires back-door adjustment

In [ ]:
mem_run = Run(
    schema_version='0.1.0',
    run_id='demo-mem-001',
    steps=[
        Step(step_index=0, decisions=[Decision(decision_id='d0', decision_type='plan_step', chosen_action='begin')]),
        Step(step_index=1, decisions=[Decision(decision_id='d1', decision_type='memory_read', chosen_action='recent_5')]),
    ],
    outcome=Outcome(kind='binary', value=False, verifier='pytest'),
)
bounded = intervene(
    dag=build_dag(mem_run),
    model=synth_model,
    step=1,
    intervention={'memory_content': 'all'},
)

print(f'identifiability : {bounded.identifiability.value}')
print(f'E-value         : {bounded.bounds.e_value:.3f}  ({bounded.bounds.technique})')
print(f'next_step       : {bounded.next_step.action}')
print(f'                  {bounded.next_step.human_text}')
print()
print('assumptions:')
for a in bounded.assumptions:
    print(f'  - {a}')
assert bounded.identifiability == IdentifiabilityStatus.BOUNDED
assert bounded.bounds is not None

### 3c. *unidentified* — `prompt_content` is replay-only

The taxonomy treats prompt-content interventions as `always-replay`: the prompt is high-dim, randomization in the corpus does not cover it, and the LLM completion is opaque. `intervene` returns `unidentified` with a structured `next_step.action='replay_required'` — the only honest answer.

In [ ]:
unidentified = intervene(
    dag=synth_dag,
    model=synth_model,
    step=2,
    intervention={'prompt_content': 'think step by step'},
)

print(f'identifiability : {unidentified.identifiability.value}')
print(f'reason          : {unidentified.reason}')
print(f'next_step.action: {unidentified.next_step.action}')
print(f'next_step.text  : {unidentified.next_step.human_text}')
print(f'next_step.payload: {unidentified.next_step.payload}')
print()
print('warnings:')
for w in unidentified.warnings:
    print(f'  ! {w}')
assert unidentified.identifiability == IdentifiabilityStatus.UNIDENTIFIED
assert unidentified.next_step.action == 'replay_required'
assert unidentified.next_step.payload['intervention_target'] == 'prompt_content'

## 4. Ranked failure attribution

Pick a failed synthetic run; rank its decisions by estimated causal influence on the outcome. Each entry inherits the per-decision identifiability label, so callers can filter or weight by epistemic confidence rather than treating all rankings as equal.

In [ ]:
failed_runs = [r for r in synth_runs if r.outcome.value is False]
print(f'failed runs in synthetic corpus: {len(failed_runs)} / {len(synth_runs)}')

case = failed_runs[0]
attribution = attribute_failure(dag=build_dag(case), model=synth_model)
top5 = attribution.top_k(5)

print(f'\nfailure attribution for {case.run_id}:')
print(f'{"rank":>4}  {"decision_id":<22}  {"type":<12}  {"action":<14}  {"influence":>9}  identifiability')
print('-' * 88)
for i, e in enumerate(top5, start=1):
    print(
        f'{i:>4}  {e.decision_id:<22}  {e.decision_type:<12}  {e.chosen_action:<14}  '
        f'{e.influence:>+9.4f}  {e.identifiability.value}'
    )
assert len(top5) <= 5

## 5. What would change the answer? — power analysis

Take a query whose CI is too wide (or whose honest verdict is `unidentified` for a non-replay reason). `power_analysis` answers the binomial-Wald question: at the current per-arm pass rates and arm fractions, what `n` would shrink the 95% CI on the marginal effect to `target_ci_width`?

Scope is deliberately narrow per design.md D2 — this is not effect-size-aware power analysis. It's the *one* helper the demo's closing paragraph needs to answer 'why don't you just collect more data?'

In [ ]:
# On the real corpus we need both arms to have observed support to make
# the question well-posed. We default to the synthetic corpus where the
# arm distribution is known, then optionally also report on the real
# corpus when both arms are present.
rep_synth = power_analysis(
    synth_runs,
    decision_type='model_call',
    arms=('sonnet', 'haiku'),
    target_ci_width=0.02,
)
print('--- synthetic corpus ---')
print(f'current_n          : {rep_synth.current_n}')
if rep_synth.current_ci_width is not None:
    print(f'current CI width   : {rep_synth.current_ci_width:.4f}')
print(f'target CI width    : {rep_synth.target_ci_width}')
print(f'estimated required n: {rep_synth.estimated_required_n}')
print()
print('assumptions:')
for a in rep_synth.assumptions:
    print(f'  - {a}')

if real_corpus:
    print()
    print('--- real corpus ---')
    rep_real = power_analysis(
        real_corpus,
        decision_type='model_call',
        arms=('small', 'large'),
        target_ci_width=0.10,
    )
    print(f'current_n          : {rep_real.current_n}')
    print(f'estimated required n: {rep_real.estimated_required_n}')
    if rep_real.warnings:
        for w in rep_real.warnings:
            print(f'  ! {w}')

## Takeaways

- The synthetic SCM canary recovers the known headline effect within ±0.05. The schema → DAG → outcome model → identifiability dispatch pipeline is **correct on a known-truth case**.
- Every `intervene` answer carries one of three labels with a contract on the rest of the result object — including a structured `next_step` whose `action` ∈ {`increase_n`, `broaden_arm_support`, `replay_required`, `add_arm_randomization`, `none`}. **No silent Pearl-L3 claims sneak in.**
- The naive marginal (`pass_rate_by_arm`) is exposed as a labeled comparison baseline so the honest causal estimator can stand next to it. **The disagreement, when it appears, is the diagnostic.**
- `power_analysis` connects 'this CI is too wide' to a concrete `n` under documented assumptions. **You can act on the answer.**

What `counterfact` is for: a tool that **says no when the data says no**, and tells you what would change the answer.